# Pencil-beam cosmic variance at $z\simeq10$ (Lumina / Lumina-EDE)

`scripts/compute_UVLF_cv.py` measures $\sigma_{\rm CV}$ by cutting the box into a
grid of **non-overlapping, axis-aligned rectangles** (`geometry="rectangle"`).
That geometry is wrong in two ways for a real survey: a survey field is a
**pencil beam** down a random line of sight, not along a box axis, so the tiling
never averages over orientation; and `floor(L/l)` fields per axis throws away
volume and fixes the field count by geometry rather than by choice.

Here every field is a randomly placed, randomly oriented box of dimensions
$(w,h,d)$, wrapped periodically (minimum image). Beams overlap, deliberately:
Szapudi & Colombi (1996) advocate high oversampling, and Colombi et al. (2000)
run $512^3$ overlapping cells. Overlap costs nothing in bias and buys
measurement precision.

## Estimator

Uniform placement in a periodic box fixes the mean **exactly, conditional on
this box**:
$$\mu_{\rm box} = N_{\rm box}(<M_{\rm UV})\,\frac{V_{\rm beam}}{L^3}.$$
This is the box's mean, not the universal mean; the offset between them is the
integral constraint and is not something we estimate. Writing
$Z_i=(N_i-\mu_{\rm box})^2$, the estimator is
$$\sigma^2_{\rm CV} = \frac{\langle Z\rangle - \mu_{\rm box}}{\mu_{\rm box}^2}.$$

The counts-in-cells literature instead uses the second factorial moment
$F_2=\frac1M\sum_i N_i(N_i-1)$, giving $\sigma^2_{\rm CV}=F_2/\mu_{\rm box}^2-1$,
which cancels shot noise with no subtraction at all (SC96). The two are **equal
in expectation** but not identical; they differ by exactly
$$(2\mu_{\rm box}-1)\,(\bar N-\mu_{\rm box})/\mu_{\rm box}^2 .$$
Since $\mathrm{Cov}(\hat\sigma^2_{\rm CV},\bar N)>0$, that extra term strictly
inflates the factorial form's variance. Measured on a clustered synthetic box,
its scatter is larger by $1.61\times$ at $\mu=3.5$, $1.34\times$ at $\mu=1.3$
and $1.07\times$ at $\mu=0.6$ — the ratio collapsing as $(2\mu-1)\to0$, as the
algebra requires. So we use the **central form as primary** and print $F_2$
alongside. $F_2$ is the right tool when the mean is *unknown*, which is the
situation SC are writing for; our $\mu_{\rm box}$ is exact by construction of
the periodic box, and the central form is what exploits it. On a lightcone or a
sub-volume, where the mean must be estimated from the fields, this ranking flips.

Overlap cannot bias either form: each beam's contribution is individually
unbiased whatever the other beams do.

## Uncertainty

Following SC96, two errors that must not be merged:

* **Measurement error** — finite sampling of *this* box. Reducible by
  oversampling. Since $\mu_{\rm box}$ is a constant,
  $$\mathrm{Var}(\hat\sigma^2_{\rm CV})
  = \frac{1}{M^2\mu_{\rm box}^4}\sum_{ij}\mathrm{Cov}(Z_i,Z_j),
  \qquad Z_i=(N_i-\mu_{\rm box})^2 .$$
  We get that double sum two ways: directly, from the scatter of repeated
  independent placements (model-free), and by modelling
  $\mathrm{Cov}(Z_i,Z_j)$ from pair geometry alone — shared volume fraction $f$
  for the overlapping branch, centre separation $r$ for the disjoint branch.
  Agreement between the two validates SC's geometry-only modelling in our box.
  The disjoint branch is **not** zero: removing the box zero mode imposes an
  integral constraint, but sub-box modes still correlate separated beams
  (positively at moderate $r$, slightly negative at the largest $r$).

* **Finite-volume error** — the box has no modes below $k_f=2\pi/L$. Not
  measurable internally at all. It comes from the continuum window integral,
  quoted separately, never folded into the measurement error bar.

Because the three $M_{\rm UV}$ limits are nested subsamples of the same beams,
their $\sigma_{\rm CV}$ estimates are strongly correlated. We report the full
cosmic cross-correlation matrix across limits (SCB99), not three independent
bars.

## Assumptions

1. Uniform random centres, Haar-uniform random orientations; all beams share
   one marginal count distribution.
2. $\mu_{\rm box}$ is exact **conditional on this box**, not the universal mean.
3. Minimum-image wrapping requires beam half-diagonal $< L/2$ (asserted).
4. **Locally Poisson** discreteness, so $F_2$ removes shot noise exactly. This
   is SC's assumption too, and it is an assumption.
5. **Ergodicity**: $\mathrm{Cov}(Z_i,Z_j)$ depends on the pair only through
   $(f,r)$ and is estimated by averaging pairs of similar geometry in one box.
   Pairs at large $r$ are few and mutually correlated.
6. **Isotropy** of that covariance. Weakest where the beam depth (369 cMpc)
   approaches the box scale, since a periodic box is not isotropic there.
7. Super-box modes are absent by construction.
8. **Linear response** for the missing-mode term,
   $\sigma^2_{\rm CV,true}\approx\sigma^2_{\rm CV,meas}+b_{\rm eff}^2(\sigma_{m,\rm cont}^2-\sigma_{m,\rm box}^2)$;
   neglects the abundance response $dn/d\delta_b$.
9. **Single snapshot**: the beam spans $z=9.2$–$10.9$ but is populated from the
   $z\simeq10$ snapshot, with no light-cone evolution along its depth.
10. $b_{\rm eff}$ from Tinker+10, `SubhaloMass` as an $M_{200c}$ proxy,
    averaged over bright galaxies box-wide rather than per beam.
11. Model variants A1/C1/B2/C2 change $M_{\rm UV}$ only; positions and halo
    masses are untouched.
12. Beam dimensions and $\sigma_m$ use each run's own cosmology.
13. The trial and pair routes estimate the same quantity and are kept as a
    mutual cross-check.

**References.** Szapudi & Colombi 1996, ApJ 470, 131 (cosmic error on $F_k$ as
a pair integral, split overlapping/disjoint); Szapudi, Colombi & Bernardeau
1999, MNRAS 310, 428 (full covariance / cross-correlation matrix); Colombi,
Szapudi, Jenkins & Colberg 2000, MNRAS 313, 711 (numerical validation, and
where the hierarchical models degrade); Colombi, Szapudi & Szalay 1998,
MNRAS 296, 253 (sampling effects).

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from scipy.spatial.transform import Rotation

repo_root = Path(".")
if Path.cwd() != repo_root:
    os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts import compute_UVLF_cv as cv
from scripts import observation_data as obs

# importing cv already applies Jacob's plot style and builds the cosmologies

In [ ]:
# ------------------------------------------------------------------ config

INSTRUMENTS = {
    "NIRCam": (4.4 * u.arcmin, 2.2 * u.arcmin),
    "Roman": (0.8 * u.deg, 0.4 * u.deg),
}

INSTRUMENT = "NIRCam"   # matches survey_area=9.7 arcmin^2 and the PANORAMIC points
N_BEAMS = 500           # beams per trial (M)
N_TRIALS = 40           # independent placement trials (R)
BASE_SEED = 20260813
SIGMA_CV_ERROR = "single_trial"  # spread of one M-beam set: the overlap-aware error bar.
                                 # "mean" divides it by sqrt(R), which is only honest if
                                 # the trials are independent -- they re-cover one box.

# pair-covariance analysis (SC96 decomposition). Baseline runs only: six full
# pair analyses is the expensive path and adds nothing the baseline does not show.
RUN_PAIR_ANALYSIS = True
N_PAIR_BEAMS = 200      # beams per pair realisation; all M(M-1)/2 pairs are used
N_PAIR_TRIALS = 20      # cost is n_trials * M(M-1)/2 * N_MC_VOLUME point tests
N_MC_VOLUME = 64        # Monte Carlo samples per pair for the shared-volume fraction
N_F_BINS = 8            # bins in shared volume fraction (overlapping branch, sparsely populated)
N_R_BINS = 12           # bins in centre separation (disjoint branch)

RUN_SATURATION = True   # measurement error vs number of beams (SC's independent-cell count)
SATURATION_BEAMS = [50, 100, 200, 400, 800]

RUN_VALIDATION = False  # geometry self-checks + old-vs-new comparison

beam_width, beam_height = INSTRUMENTS[INSTRUMENT]

# everything else is inherited from compute_UVLF_cv so the two scripts cannot drift
z_min, z_max, z_center = cv.z_min, cv.z_max, cv.z_center
MUV_limits = cv.MUV_limits
MUV_bin_edges, MUV_bin_width = cv.MUV_bin_edges, cv.MUV_bin_width
snap_num = cv.snap_num
lcdm_base_path = cv.lcdm_base_path
ede_hdf5_path = cv.ede_hdf5_path
redshift_EDE = cv.redshift_EDE
box_size_cmpc_EDE = cv.box_size_cmpc_EDE
random_seed = cv.random_seed
best_delta_MUV = cv.best_delta_MUV
best_sigma_UV = cv.best_sigma_UV
best_sfe_slope = cv.best_sfe_slope
best_sigma_UV_norm = cv.best_sigma_UV_norm
obs_mu, obs_sigma_cv = cv.obs_mu, cv.obs_sigma_cv
obs_lower_error, obs_upper_error = cv.obs_lower_error, cv.obs_upper_error

output_dir = Path("outputs") / f"pencilbeam_{INSTRUMENT}"
output_dir.mkdir(parents=True, exist_ok=True)

print("instrument:", INSTRUMENT, beam_width, "x", beam_height)
print("beams per trial:", N_BEAMS, "trials:", N_TRIALS)
print("output directory:", output_dir)

In [ ]:
# ------------------------------------------------------------------ geometry

def beam_dimensions(cosmology, z_center, z_min, z_max, width_angle, height_angle):
    """Comoving (width, height, depth) of one survey field, in cMpc."""
    return cv.calculate_rectangle_field_lengths(
        cosmology, z_center, z_min, z_max, width_angle, height_angle,
    )

def place_beams(n_beams, box_size, seed=42):
    """Uniform random centres in the box, Haar-uniform random orientations."""
    rng = np.random.default_rng(seed)
    centers = rng.random((n_beams, 3)) * box_size

    try:
        frames = Rotation.random(n_beams, rng=rng).as_matrix()
    except TypeError:  # scipy < 1.15 does not accept rng=
        frames = Rotation.random(n_beams, random_state=seed).as_matrix()

    return centers, frames

def check_beam_fits(dims, box_size):
    """Minimum-image wrapping is only valid if the beam fits inside half the box."""
    half_diagonal = 0.5 * np.sqrt(np.sum(np.asarray(dims, dtype=float) ** 2))

    print("beam dimensions [cMpc]:", dims)
    print("beam volume [cMpc^3]:", np.prod(dims))
    print("beam half diagonal:", half_diagonal, "half box:", box_size / 2)

    if half_diagonal >= box_size / 2:
        raise ValueError("beam is too large for minimum-image wrapping")

    return half_diagonal

def minimum_image(displacement, box_size):
    """Wrap separations into [-L/2, L/2). Valid because the beam fits in half a box."""
    return (displacement + box_size / 2) % box_size - box_size / 2

def galaxies_in_beam(positions, center, frame, dims, box_size):
    """Indices of galaxies inside one randomly oriented beam.

    positions must already be wrapped into [0, box_size). frame columns are the
    width, height and line-of-sight unit vectors, so local = displacement @ frame.
    """
    w, h, d = dims

    displacement = positions - center
    displacement[displacement > box_size / 2] -= box_size
    displacement[displacement < -box_size / 2] += box_size

    field_coordinates = displacement @ frame

    inside = (
        (field_coordinates[:, 0] >= -w / 2)
        & (field_coordinates[:, 0] < w / 2)
        & (field_coordinates[:, 1] >= -h / 2)
        & (field_coordinates[:, 1] < h / 2)
        & (field_coordinates[:, 2] >= -d / 2)
        & (field_coordinates[:, 2] < d / 2)
    )

    return np.where(inside)[0]

In [ ]:
# ------------------------------------------------- counts, mu_box, factorial moments

def beam_count_matrix(positions, MUV, MUV_limits, centers, frames, dims, box_size, verbose=False):
    """(n_beams, n_limits) matrix of galaxy counts brighter than each MUV limit."""
    positions = np.asarray(positions, dtype=float)
    MUV = np.asarray(MUV, dtype=float)
    MUV_limits = np.asarray(MUV_limits, dtype=float)

    assert len(positions) == len(MUV)
    assert len(centers) == len(frames)

    # only galaxies that can ever be counted need to be tested against the beams
    faintest_limit = np.max(MUV_limits)
    candidate = np.isfinite(MUV) & (MUV < faintest_limit)
    candidate_positions = np.ascontiguousarray(positions[candidate])
    candidate_MUV = MUV[candidate]

    n_beams = len(centers)
    n_limits = len(MUV_limits)
    count_matrix = np.zeros((n_beams, n_limits), dtype=int)

    for field_id in range(n_beams):
        galaxy_ids = galaxies_in_beam(
            positions=candidate_positions,
            center=centers[field_id],
            frame=frames[field_id],
            dims=dims,
            box_size=box_size,
        )
        field_MUV = candidate_MUV[galaxy_ids]

        for limit_id in range(n_limits):
            count_matrix[field_id, limit_id] = np.count_nonzero(field_MUV < MUV_limits[limit_id])

    if verbose:
        print("number of beams:", n_beams)
        print("candidate galaxies:", np.count_nonzero(candidate))
        for limit_id in range(n_limits):
            print(
                "MUV limit:", MUV_limits[limit_id],
                "mean count:", np.mean(count_matrix[:, limit_id]),
                "empty beams:", np.count_nonzero(count_matrix[:, limit_id] == 0),
            )

    return count_matrix

def mu_box(MUV, MUV_limits, dims, box_size):
    """Exact mean count per beam, conditional on this box: mu = N_box * V_beam / L^3.

    Uniform placement in a periodic box makes this exact, so nothing downstream
    estimates a mean. It is the box mean, not the universal mean.
    """
    MUV = np.asarray(MUV, dtype=float)
    volume_fraction = np.prod(dims) / box_size ** 3

    counts = np.array(
        [np.count_nonzero(np.isfinite(MUV) & (MUV < MUV_limit)) for MUV_limit in MUV_limits],
        dtype=float,
    )

    return counts * volume_fraction

def central_second_moment(count_matrix, expected_means):
    """Z_i = (N_i - mu_box)^2 per beam. The primary estimator's per-beam quantity."""
    counts = np.asarray(count_matrix, dtype=float)
    return (counts - np.asarray(expected_means, dtype=float)[None, :]) ** 2

def second_factorial_moment(count_matrix):
    """Z_i = N_i (N_i - 1) per beam, and F_2 = <Z>. Shot noise cancels in Z (SC96)."""
    counts = np.asarray(count_matrix, dtype=float)
    return counts * (counts - 1.0)

def sigma_cv_squared(count_matrix, expected_means):
    """sigma_CV^2 = (<(N - mu_box)^2> - mu_box) / mu_box^2. Primary estimator.

    Equal in expectation to the factorial form F_2/mu_box^2 - 1, but strictly more
    precise here, because mu_box is known exactly and this form centres every term
    on it. The two differ by exactly (2 mu_box - 1)(Nbar - mu_box)/mu_box^2, so the
    factorial form lets the sample-mean fluctuation leak in, scaled by (2 mu - 1).
    Measured on a clustered synthetic box, the factorial form's scatter is larger
    by 1.61x at mu=3.5, 1.34x at mu=1.3, 1.07x at mu=0.6 -- the ratio collapsing
    as (2 mu - 1) -> 0, as the algebra requires.

    This ranking depends on mu_box being exact. On a lightcone or a sub-volume
    where the mean must be estimated from the fields, F_2 is the correct choice
    and this one is not available.
    """
    expected_means = np.asarray(expected_means, dtype=float)
    Z = np.mean(central_second_moment(count_matrix, expected_means), axis=0)
    return (Z - expected_means) / expected_means ** 2

def sigma_cv_squared_factorial(count_matrix, expected_means):
    """sigma_CV^2 = F_2 / mu_box^2 - 1, the counts-in-cells form, for comparison."""
    F2 = np.mean(second_factorial_moment(count_matrix), axis=0)
    return F2 / np.asarray(expected_means, dtype=float) ** 2 - 1.0

def validate_mean_counts(count_matrix, MUV, MUV_limits, dims, box_size):
    """Check the measured mean count against the exact expectation."""
    expected_means = mu_box(MUV, MUV_limits, dims, box_size)
    measured_means = np.mean(count_matrix, axis=0)
    mean_errors = np.zeros(len(MUV_limits))

    if len(count_matrix) > 1:
        mean_errors = np.std(count_matrix, axis=0, ddof=1) / np.sqrt(len(count_matrix))

    for limit_id, MUV_limit in enumerate(MUV_limits):
        print("MUV limit:", MUV_limit)
        print("  mu_box (exact):", expected_means[limit_id])
        print("  measured mean:", measured_means[limit_id])
        print("  mean uncertainty:", mean_errors[limit_id])

        if mean_errors[limit_id] > 0:
            print("  difference in standard errors:",
                  (measured_means[limit_id] - expected_means[limit_id]) / mean_errors[limit_id])

    return expected_means, measured_means, mean_errors

In [ ]:
# ------------------------------------------------- measurement error from trials

def beam_cv_trials(
    positions,
    MUV,
    MUV_limits,
    dims,
    box_size,
    n_beams=N_BEAMS,
    n_trials=N_TRIALS,
    base_seed=BASE_SEED,
    model_name="",
    error_mode=SIGMA_CV_ERROR,
):
    """Repeat the M-beam measurement R times; the trial scatter is the measurement error.

    Var(F_2_hat) = (1/M^2) sum_ij Cov(Z_i, Z_j) exactly, so the trial-to-trial
    scatter *is* that double sum -- no model of the pair covariance is needed for
    the error bar itself. `pair_covariance_analysis` decomposes the same sum into
    SC's overlapping and disjoint branches, and the two must agree.

    Overlapping beams are not independent, so M_eff = Var(Z)/Var(F_2_hat) < M is
    reported: the number of independent beams the M overlapping ones are worth.
    All trial values are kept; error bars use the complete set, not percentiles.
    """
    positions = np.asarray(positions, dtype=float)
    MUV = np.asarray(MUV, dtype=float)
    MUV_limits = np.asarray(MUV_limits, dtype=float)

    expected_means = mu_box(MUV, MUV_limits, dims, box_size)
    n_limits = len(MUV_limits)

    Z_trials = np.empty((n_trials, n_limits))
    F2_trials = np.empty((n_trials, n_limits))
    mu_trials = np.empty((n_trials, n_limits))
    empty_fraction = np.empty((n_trials, n_limits))

    # pooled moments of Z, for the effective-beam-count diagnostic
    z_sum = np.zeros(n_limits)
    z_squared_sum = np.zeros(n_limits)

    start_time = time.time()

    for trial_id in range(n_trials):
        centers, frames = place_beams(n_beams=n_beams, box_size=box_size, seed=base_seed + trial_id)

        counts = beam_count_matrix(
            positions=positions, MUV=MUV, MUV_limits=MUV_limits,
            centers=centers, frames=frames, dims=dims, box_size=box_size, verbose=False,
        )

        Z = central_second_moment(counts, expected_means)

        Z_trials[trial_id] = np.mean(Z, axis=0)
        F2_trials[trial_id] = np.mean(second_factorial_moment(counts), axis=0)
        mu_trials[trial_id] = np.mean(counts, axis=0)
        empty_fraction[trial_id] = np.mean(counts == 0, axis=0)

        z_sum += np.sum(Z, axis=0)
        z_squared_sum += np.sum(Z ** 2, axis=0)

    # sigma_CV^2 = (<Z> - mu_box)/mu_box^2, one value per trial, complete set retained
    sigma_cv2_trials = (Z_trials - expected_means[None, :]) / expected_means[None, :] ** 2
    sigma_cv2_factorial_trials = F2_trials / expected_means[None, :] ** 2 - 1.0
    sigma_cv2 = np.mean(sigma_cv2_trials, axis=0)
    sigma_cv = np.sqrt(np.clip(sigma_cv2, 0.0, None))

    trial_scatter = np.std(sigma_cv2_trials, axis=0, ddof=1) if n_trials > 1 else np.zeros(n_limits)
    mu_scatter = np.std(mu_trials, axis=0, ddof=1) if n_trials > 1 else np.zeros(n_limits)

    if error_mode == "mean":
        sigma_cv2_error = trial_scatter / np.sqrt(n_trials)
        mu_error = mu_scatter / np.sqrt(n_trials)
    elif error_mode == "single_trial":
        sigma_cv2_error = trial_scatter
        mu_error = mu_scatter
    else:
        raise ValueError("error_mode must be 'mean' or 'single_trial'")

    # full cosmic cross-correlation matrix across MUV limits (SCB99): the limits
    # are nested subsamples of the same beams, so these are far from independent
    covariance_matrix = np.cov(sigma_cv2_trials, rowvar=False) if n_trials > 1 else np.full((n_limits, n_limits), np.nan)
    covariance_matrix = np.atleast_2d(covariance_matrix)
    standard_deviations = np.sqrt(np.clip(np.diag(covariance_matrix), 0.0, None))
    with np.errstate(divide="ignore", invalid="ignore"):
        correlation_matrix = covariance_matrix / np.outer(standard_deviations, standard_deviations)

    # how many independent beams the M overlapping beams are worth
    n_total = n_beams * n_trials
    z_mean = z_sum / n_total
    z_variance = np.clip(z_squared_sum / n_total - z_mean ** 2, 0.0, None)
    Z_mean_variance = np.var(Z_trials, axis=0, ddof=1) if n_trials > 1 else np.full(n_limits, np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        n_beams_effective = np.where(Z_mean_variance > 0, z_variance / Z_mean_variance, np.nan)
        overlap_ratio = n_beams / n_beams_effective

    beam_volume_fraction = n_beams * np.prod(dims) / box_size ** 3

    # sigma_CV^2 +/- error propagated into asymmetric errors on sigma_CV
    sigma_cv_lower = np.sqrt(np.clip(sigma_cv2 - sigma_cv2_error, 0.0, None))
    sigma_cv_upper = np.sqrt(np.clip(sigma_cv2 + sigma_cv2_error, 0.0, None))

    if model_name:
        print(f"\n{model_name}: pencil-beam cosmic variance")
        print(f"  {n_trials} trials x {n_beams} beams in {time.time() - start_time:.1f} s")
        print(f"  beam volume per trial / box volume: {beam_volume_fraction:.3f}")
        for limit_id, MUV_limit in enumerate(MUV_limits):
            print(
                f"  M_UV < {MUV_limit}: "
                f"mu_box={expected_means[limit_id]:.4g} (measured {np.mean(mu_trials[:, limit_id]):.4g}), "
                f"sigma_cv={sigma_cv[limit_id]:.4g} "
                f"(-{sigma_cv[limit_id] - sigma_cv_lower[limit_id]:.3g} "
                f"+{sigma_cv_upper[limit_id] - sigma_cv[limit_id]:.3g}), "
                f"empty beams={np.mean(empty_fraction[:, limit_id]):.3f}, "
                f"M_eff={n_beams_effective[limit_id]:.4g} of {n_beams} "
                f"(overlap ratio {overlap_ratio[limit_id]:.2f})"
            )
        factorial_sigma_cv = np.sqrt(np.clip(np.mean(sigma_cv2_factorial_trials, axis=0), 0.0, None))
        print("  factorial form F_2/mu^2-1 (equal in expectation, noisier):",
              np.array2string(factorial_sigma_cv, precision=4, floatmode="fixed"),
              " its scatter /central scatter:",
              np.array2string(np.std(sigma_cv2_factorial_trials, axis=0, ddof=1) / trial_scatter,
                              precision=3, floatmode="fixed"))
        print("  cross-correlation of sigma_CV^2 between MUV limits (SCB99):")
        for row in np.atleast_2d(correlation_matrix):
            print("   ", np.array2string(row, precision=3, floatmode="fixed"))

    return {
        "expected_means": expected_means,
        "measured_means": np.mean(mu_trials, axis=0),
        "mu_error": mu_error,
        "sigma_cv_squared": sigma_cv2,
        "sigma_cv_squared_error": sigma_cv2_error,
        "sigma_cv": sigma_cv,
        "sigma_cv_lower": sigma_cv_lower,
        "sigma_cv_upper": sigma_cv_upper,
        "empty_fraction": np.mean(empty_fraction, axis=0),
        "sigma_cv2_trials": sigma_cv2_trials,
        "sigma_cv2_factorial_trials": sigma_cv2_factorial_trials,
        "mu_trials": mu_trials,
        "covariance_matrix": covariance_matrix,
        "correlation_matrix": correlation_matrix,
        "z_variance": z_variance,
        "Z_mean_variance": Z_mean_variance,
        "n_beams_effective": n_beams_effective,
        "overlap_ratio": overlap_ratio,
        "n_beams": n_beams,
        "n_trials": n_trials,
    }

In [ ]:
# ------------------------------------- pair covariance: SC96 overlapping / disjoint

def shared_volume_fraction(centers, frames, pair_i, pair_j, dims, box_size, n_mc=N_MC_VOLUME, seed=0):
    """Monte Carlo estimate of V_intersection / V_beam for each listed pair.

    Two arbitrarily oriented boxes: sample uniformly inside beam i, count the
    fraction landing inside beam j. Chunked so memory stays bounded.

    The overlap is periodic, and at these dimensions that is not a technicality:
    the beam depth (369 cMpc) is 74 per cent of the box, so two beams can share
    volume through the wrap as well as directly. Two axis-aligned beams offset by
    half a depth share 0.645 of their volume, not 0.5. This is assumption 6
    biting -- the box is not isotropic on the scale of the beam's long axis.
    """
    dims = np.asarray(dims, dtype=float)
    rng = np.random.default_rng(seed)
    n_pairs = len(pair_i)
    fraction = np.empty(n_pairs)

    chunk_size = max(1, int(2e6 // n_mc))

    for start in range(0, n_pairs, chunk_size):
        stop = min(start + chunk_size, n_pairs)
        i = pair_i[start:stop]
        j = pair_j[start:stop]

        # uniform samples inside beam i, in i's local frame
        local = (rng.random((stop - start, n_mc, 3)) - 0.5) * dims
        # local -> world: x = center_i + frame_i @ local
        world = centers[i][:, None, :] + np.einsum("pab,pnb->pna", frames[i], local)

        displacement = minimum_image(world - centers[j][:, None, :], box_size)
        # world -> j's local frame
        local_j = np.einsum("pna,pab->pnb", displacement, frames[j])

        inside = np.all(np.abs(local_j) < dims / 2, axis=2)
        fraction[start:stop] = np.mean(inside, axis=1)

    return fraction

def pair_covariance_analysis(
    positions,
    MUV,
    MUV_limits,
    dims,
    box_size,
    n_beams=N_PAIR_BEAMS,
    n_trials=N_PAIR_TRIALS,
    base_seed=BASE_SEED + 9000,
    n_f_bins=N_F_BINS,
    n_r_bins=N_R_BINS,
    model_name="",
):
    """Decompose sum_ij Cov(Z_i,Z_j) into SC96's overlapping and disjoint branches.

    For every pair we measure the shared volume fraction f and the centre
    separation r, then bin the products (Z_i - Zbar)(Z_j - Zbar). Binning replaces
    an ensemble average over realisations of a fixed pair with a spatial average
    over many pairs of similar geometry -- the ergodicity assumption (5).

    The binned model is then summed back over the true placement and compared
    with the raw double sum. Agreement means pair geometry alone (f, r) captures
    the covariance structure, which is SC's modelling assumption, tested here
    rather than assumed.
    """
    MUV_limits = np.asarray(MUV_limits, dtype=float)
    n_limits = len(MUV_limits)
    expected_means = mu_box(MUV, MUV_limits, dims, box_size)

    pair_i, pair_j = np.triu_indices(n_beams, k=1)
    max_separation = 0.5 * np.sqrt(3.0) * box_size

    f_edges = np.linspace(0.0, 1.0, n_f_bins + 1)
    r_edges = np.linspace(0.0, max_separation, n_r_bins + 1)

    start_time = time.time()

    # ---- pass 1: counts and pair geometry for every trial.
    # Z must be centred on the GLOBAL mean, never the per-trial mean: subtracting
    # the within-trial mean forces sum_i dZ_i = 0, which makes the double sum
    # sum_ij dZ_i dZ_j vanish identically. That is the integral constraint
    # reappearing inside the estimator, and it silently destroys the measurement.
    Z_trials, fraction_trials, separation_trials = [], [], []

    for trial_id in range(n_trials):
        centers, frames = place_beams(n_beams=n_beams, box_size=box_size, seed=base_seed + trial_id)

        counts = beam_count_matrix(
            positions=positions, MUV=MUV, MUV_limits=MUV_limits,
            centers=centers, frames=frames, dims=dims, box_size=box_size, verbose=False,
        )
        Z_trials.append(central_second_moment(counts, expected_means))

        separation_trials.append(np.linalg.norm(
            minimum_image(centers[pair_i] - centers[pair_j], box_size), axis=1
        ))
        fraction_trials.append(shared_volume_fraction(
            centers, frames, pair_i, pair_j, dims, box_size, seed=base_seed + trial_id,
        ))

    Z_mean = np.mean(np.concatenate(Z_trials, axis=0), axis=0)

    # ---- pass 2: bin the pair products by geometry
    def accumulate(trial_ids):
        f_sum = np.zeros((n_f_bins, n_limits))
        r_sum = np.zeros((n_r_bins, n_limits))
        f_count = np.zeros(n_f_bins)
        r_count = np.zeros(n_r_bins)
        diagonal = np.zeros(n_limits)
        offdiagonal = np.zeros(n_limits)
        bins = []

        for trial_id in trial_ids:
            dZ = Z_trials[trial_id] - Z_mean
            products = dZ[pair_i] * dZ[pair_j]

            fraction = fraction_trials[trial_id]
            separation = separation_trials[trial_id]
            overlapping = fraction > 0

            f_bin = np.clip(np.digitize(fraction[overlapping], f_edges) - 1, 0, n_f_bins - 1)
            r_bin = np.clip(np.digitize(separation[~overlapping], r_edges) - 1, 0, n_r_bins - 1)

            for limit_id in range(n_limits):
                f_sum[:, limit_id] += np.bincount(f_bin, weights=products[overlapping, limit_id], minlength=n_f_bins)
                r_sum[:, limit_id] += np.bincount(r_bin, weights=products[~overlapping, limit_id], minlength=n_r_bins)

            f_count += np.bincount(f_bin, minlength=n_f_bins)
            r_count += np.bincount(r_bin, minlength=n_r_bins)

            diagonal += np.sum(dZ ** 2, axis=0)
            offdiagonal += 2.0 * np.sum(products, axis=0)
            bins.append((overlapping, f_bin, r_bin))

        n = len(trial_ids)
        with np.errstate(divide="ignore", invalid="ignore"):
            f_profile = f_sum / f_count[:, None]
            r_profile = r_sum / r_count[:, None]

        return {
            "f_profile": f_profile, "r_profile": r_profile,
            "f_count": f_count, "r_count": r_count,
            "diagonal": diagonal / n, "offdiagonal": offdiagonal / n,
            "overlapping_total": 2.0 * np.nansum(f_profile * f_count[:, None], axis=0) / n,
            "disjoint_total": 2.0 * np.nansum(r_profile * r_count[:, None], axis=0) / n,
            "bins": bins,
        }

    full = accumulate(range(n_trials))

    diagonal = full["diagonal"]
    raw_offdiagonal = full["offdiagonal"]
    overlapping_total = full["overlapping_total"]
    disjoint_total = full["disjoint_total"]
    f_profile, r_profile = full["f_profile"], full["r_profile"]
    f_count, r_count = full["f_count"], full["r_count"]

    total_sum = diagonal + raw_offdiagonal
    Z_mean_variance = total_sum / n_beams ** 2
    # sigma_CV^2 = (<Z> - mu)/mu^2 and mu is a constant, so Var(sigma_CV^2) = Var(<Z>)/mu^4
    sigma_cv2_variance = Z_mean_variance / expected_means ** 4

    # exact bookkeeping check: Var(<Z>) = (1/M^2) sum_ij Cov(Z_i,Z_j) is an
    # identity, so the pair decomposition must reproduce the trial variance of
    # <Z> to machine precision. This validates the branch split, unlike the
    # split-half test below which is genuinely noisy.
    Z_per_trial = np.array([np.mean(Z, axis=0) for Z in Z_trials])
    Z_variance_direct = np.mean((Z_per_trial - Z_mean) ** 2, axis=0)
    identity_ratio = Z_mean_variance / Z_variance_direct

    with np.errstate(divide="ignore", invalid="ignore"):
        n_beams_effective = np.where(total_sum > 0, n_beams * diagonal / total_sum, np.nan)

    # ---- split-half test of the geometry-only model.
    # Calibrating and evaluating on the same pairs is circular and reproduces the
    # raw sum exactly. Calibrate C(f, r) on the first half of the trials, predict
    # the second half's double sum, compare against what the second half measured.
    half = n_trials // 2
    model_ratio = np.full(n_limits, np.nan)

    if half >= 1 and n_trials - half >= 1:
        calibration = accumulate(range(half))
        held_out = accumulate(range(half, n_trials))

        predicted = np.zeros(n_limits)
        for overlapping, f_bin, r_bin in held_out["bins"]:
            for limit_id in range(n_limits):
                f_model = np.nan_to_num(calibration["f_profile"][:, limit_id])
                r_model = np.nan_to_num(calibration["r_profile"][:, limit_id])
                predicted[limit_id] += 2.0 * (f_model[f_bin].sum() + r_model[r_bin].sum())
        predicted /= len(held_out["bins"])

        with np.errstate(divide="ignore", invalid="ignore"):
            model_ratio = predicted / held_out["offdiagonal"]

    if model_name:
        print(f"\n{model_name}: pair-covariance decomposition ({n_trials} x {n_beams} beams, "
              f"{n_trials * len(pair_i):,} pairs, {time.time() - start_time:.1f} s)")
        overlapping_pair_fraction = f_count.sum() / max(f_count.sum() + r_count.sum(), 1)
        print(f"  overlapping pairs: {overlapping_pair_fraction:.4f} of all pairs "
              f"({int(f_count.sum()):,} of {int(f_count.sum() + r_count.sum()):,})")
        for limit_id, MUV_limit in enumerate(MUV_limits):
            total = total_sum[limit_id]
            print(
                f"  M_UV < {MUV_limit}: "
                f"diagonal={diagonal[limit_id] / total:.3f}, "
                f"overlapping={overlapping_total[limit_id] / total:.3f}, "
                f"disjoint={disjoint_total[limit_id] / total:.3f} of the total sum; "
                f"M_eff={n_beams_effective[limit_id]:.4g} of {n_beams}"
            )
        print("  pair sum / trial variance of <Z> (exact identity, must be 1):",
              np.array2string(identity_ratio, precision=6, floatmode="fixed"))
        print("  split-half model/measured off-diagonal:",
              np.array2string(model_ratio, precision=2, floatmode="fixed"))
        print("   -- tests whether (f, r) alone captures the covariance, but the")
        print("      off-diagonal is a small residual of many noisy pair products,")
        print("      so this needs many trials before it means anything.")
        if np.any(n_beams_effective > n_beams):
            print("  note: M_eff > M because the disjoint branch is negative. Distant")
            print("        beams anticorrelate under the integral constraint, which")
            print("        makes the estimator better than independent sampling.")

    return {
        "f_edges": f_edges,
        "r_edges": r_edges,
        "f_profile": f_profile,
        "r_profile": r_profile,
        "f_count": f_count,
        "r_count": r_count,
        "diagonal": diagonal,
        "overlapping_total": overlapping_total,
        "disjoint_total": disjoint_total,
        "raw_offdiagonal": raw_offdiagonal,
        "model_ratio": model_ratio, "identity_ratio": identity_ratio,
        "total_sum": total_sum,
        "Z_mean_variance": Z_mean_variance,
        "sigma_cv_squared_variance": sigma_cv2_variance,
        "n_beams_effective": n_beams_effective,
        "n_beams": n_beams,
        "n_trials": n_trials,
    }

In [ ]:
# --------------------------------------- finite-volume term: box modes vs continuum

def sigma_m_modes(cosmology, dims, box_size, h, redshift, nmax=128):
    """Discrete box-mode sum of P(k)|W(k)|^2, matching cv.compute_sigma_m.

    Reimplemented over the full mode cube so the same routine can be reused for
    the continuum comparison below.
    """
    lengths = np.asarray(dims, dtype=float) * h
    box_length = float(box_size) * h
    k_fundamental = 2 * np.pi / box_length

    modes = np.arange(-nmax, nmax + 1, dtype=float)
    kx, ky, kz = np.meshgrid(modes, modes, modes, indexing="ij")

    k = k_fundamental * np.sqrt(kx ** 2 + ky ** 2 + kz ** 2)
    window = (
        np.sinc(kx * lengths[0] / box_length)
        * np.sinc(ky * lengths[1] / box_length)
        * np.sinc(kz * lengths[2] / box_length)
    )

    nonzero = k > 0
    power = cosmology.matterPowerSpectrum(k[nonzero], z=redshift, model="eisenstein98")

    return float(np.sum(power * window[nonzero] ** 2) / box_length ** 3)

def sigma_m_missing_modes(cosmology, dims, box_size, h, redshift, n_grid=128):
    """Variance in modes longer than the box: the finite-volume term.

    The box holds modes only at k = k_f n, n non-zero, with k_f = 2 pi / L. Below
    k_f it holds nothing at all, so the power a real survey sees there is simply
    absent, and no internal estimator can recover it:

        sigma_missing^2 = (2 pi)^-3 * integral over |k| < k_f of P(k) |W(k)|^2 d3k

    Positive by construction, no cancellation between large numbers. Note this is
    a LOWER BOUND on the full continuum-minus-lattice difference: it ignores the
    (smaller, either-signed) difference between the continuum integral and the
    mode sum above k_f. Quote it separately; never fold it into the measurement
    error, and never put it in the ratio plot, whose measurement side is itself
    box-truncated.
    """
    lengths = np.asarray(dims, dtype=float) * h
    box_length = float(box_size) * h
    k_fundamental = 2 * np.pi / box_length

    axis = np.linspace(-k_fundamental, k_fundamental, n_grid)
    spacing = axis[1] - axis[0]
    kx, ky, kz = np.meshgrid(axis, axis, axis, indexing="ij")
    k = np.sqrt(kx ** 2 + ky ** 2 + kz ** 2)

    # numpy sinc(x) = sin(pi x)/(pi x), so W(k) = sin(k a / 2)/(k a / 2) = sinc(k a / 2 pi)
    window_squared = (
        np.sinc(kx * lengths[0] / (2 * np.pi))
        * np.sinc(ky * lengths[1] / (2 * np.pi))
        * np.sinc(kz * lengths[2] / (2 * np.pi))
    ) ** 2

    inside = (k > 0) & (k < k_fundamental)
    power = cosmology.matterPowerSpectrum(k[inside], z=redshift, model="eisenstein98")
    missing = float(np.sum(power * window_squared[inside]) * spacing ** 3 / (2 * np.pi) ** 3)

    print(f"  k_fundamental: {k_fundamental:.5g} h/Mpc, "
          f"missing variance: {missing:.5g} (sigma = {np.sqrt(missing):.5g})")

    return missing

In [ ]:
# ------------------------------------------------- one UV model measured with beams

def beam_UV_model(
    positions,
    MUV,
    subhalo_mass,
    redshift,
    MUV_limits,
    dims,
    box_size,
    model_name,
    parameter=0.0,
    n_beams=N_BEAMS,
    n_trials=N_TRIALS,
    base_seed=BASE_SEED,
):
    """Pencil-beam replacement for og_UV_model / A1_delta_MUV / C1_sigma_UV.

    Returns the same list-of-dict schema the plotting code expects, so the model
    variants differ only by the MUV array handed in.
    """
    MUV = np.asarray(MUV, dtype=float)
    subhalo_mass = np.asarray(subhalo_mass, dtype=float)

    trials = beam_cv_trials(
        positions=positions, MUV=MUV, MUV_limits=MUV_limits, dims=dims, box_size=box_size,
        n_beams=n_beams, n_trials=n_trials, base_seed=base_seed, model_name=model_name,
    )

    results = []

    for limit_id, MUV_limit in enumerate(MUV_limits):
        bright = np.isfinite(MUV) & (MUV < MUV_limit)
        b_eff = cv.compute_linear_bias(subhalo_mass[bright], redshift)

        mu = trials["expected_means"][limit_id]
        sigma_cv = trials["sigma_cv"][limit_id]

        results.append(
            {
                "model": model_name,
                "parameter": parameter,
                "MUV_limit": MUV_limit,
                "mu": mu,
                "b_eff": b_eff,
                "sigma_cv": sigma_cv,
                "sigma_cv_squared": trials["sigma_cv_squared"][limit_id],
                "total_galaxies": int(np.count_nonzero(bright)),
                "empty_fraction": trials["empty_fraction"][limit_id],
                "n_beams_effective": trials["n_beams_effective"][limit_id],
                "mu_err_low": trials["mu_error"][limit_id],
                "mu_err_high": trials["mu_error"][limit_id],
                "sigma_cv_err_low": max(sigma_cv - trials["sigma_cv_lower"][limit_id], 0.0),
                "sigma_cv_err_high": max(trials["sigma_cv_upper"][limit_id] - sigma_cv, 0.0),
            }
        )

    return results, trials

def compute_box_UVLF(MUV_values, box_size_cmpc, MUV_bin_edges, MUV_bin_width):
    """UVLF from the whole box.

    The mean luminosity function does not depend on field geometry, so it is
    measured on the full volume rather than by stacking overlapping beams, which
    would double-count the overlap volume in the normalisation.
    """
    MUV_values = np.asarray(MUV_values, dtype=float)
    counts, _ = np.histogram(MUV_values[np.isfinite(MUV_values)], bins=MUV_bin_edges)

    phi = counts / (box_size_cmpc ** 3 * MUV_bin_width)
    MUV_bin_centers = 0.5 * (MUV_bin_edges[:-1] + MUV_bin_edges[1:])
    nonzero = phi > 0

    return MUV_bin_centers[nonzero], np.log10(phi[nonzero])

def saturation_curve(positions, MUV, MUV_limits, dims, box_size,
                     beam_counts=SATURATION_BEAMS, n_trials=N_TRIALS,
                     base_seed=BASE_SEED + 4000, model_name=""):
    """Measurement error vs number of beams: SC's independent-cell count, measured.

    Below the plateau more beams help; above it the box, not the sampling, is the
    limit and oversampling buys nothing.
    """
    errors = []
    effective = []

    for n_beams in beam_counts:
        trials = beam_cv_trials(
            positions=positions, MUV=MUV, MUV_limits=MUV_limits, dims=dims, box_size=box_size,
            n_beams=n_beams, n_trials=n_trials, base_seed=base_seed, model_name="",
            error_mode="single_trial",
        )
        errors.append(trials["sigma_cv_squared_error"])
        effective.append(trials["n_beams_effective"])
        print(f"  M={n_beams}: sigma_CV^2 error={np.array2string(trials['sigma_cv_squared_error'], precision=4)}, "
              f"M_eff={np.array2string(trials['n_beams_effective'], precision=1)}")

    if model_name:
        print(f"{model_name}: saturation scan complete")

    return np.asarray(beam_counts), np.asarray(errors), np.asarray(effective)

In [ ]:
# ------------------------------------------------------------------ plots

def _series(lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results, lcdm_C2_results,
            ede_results, best_b2_sfe_peak, c2_color="tab:brown"):
    return [
        (lcdm_results, "LCDM", "s", "gray"),
        (lcdm_A1_results, rf"LCDM A1:$\Delta M_{{UV}}={best_delta_MUV}$", "D", "tab:blue"),
        (lcdm_C1_results, rf"LCDM C1:$\sigma_{{UV}}={best_sigma_UV}$", "o", "tab:orange"),
        (lcdm_B2_results, rf"LCDM B2:$q={best_sfe_slope},\mathrm{{SFE}}_{{peak}}={best_b2_sfe_peak:.2f}$", "P", "tab:purple"),
        (lcdm_C2_results, rf"LCDM C2:$\sigma_{{UV,norm}}={best_sigma_UV_norm}$", "X", c2_color),
        (ede_results, "EDE", "^", "tab:green"),
    ]

def _ordered(results, MUV_limits):
    return [next(r for r in results if r["MUV_limit"] == MUV_limit) for MUV_limit in MUV_limits]

def plot_uvlf_overlay(lcdm_MUV, lcdm_MUV_A1, lcdm_MUV_C1, lcdm_MUV_B2, lcdm_MUV_C2, ede_MUV,
                      lcdm_box_size, ede_box_size, best_b2_sfe_peak):
    series = [
        (lcdm_MUV, lcdm_box_size, "LCDM", "s", "gray"),
        (lcdm_MUV_A1, lcdm_box_size, rf"LCDM A1:$\Delta M_{{UV}}={best_delta_MUV}$", "D", "tab:blue"),
        (lcdm_MUV_C1, lcdm_box_size, rf"LCDM C1:$\sigma_{{UV}}={best_sigma_UV}$", "o", "tab:orange"),
        (lcdm_MUV_B2, lcdm_box_size, rf"LCDM B2:$q={best_sfe_slope},\mathrm{{SFE}}_{{peak}}={best_b2_sfe_peak:.2f}$", "P", "tab:purple"),
        (lcdm_MUV_C2, lcdm_box_size, rf"LCDM C2:$\sigma_{{UV,norm}}={best_sigma_UV_norm}$", "X", "tab:brown"),
        (ede_MUV, ede_box_size, "EDE", "^", "tab:green"),
    ]

    fig, ax = plt.subplots(figsize=(9, 7))

    for MUV_values, box_size, label, marker, color in series:
        MUV_bins, log_phi = compute_box_UVLF(MUV_values, box_size, MUV_bin_edges, MUV_bin_width)
        ax.plot(MUV_bins, log_phi, marker=marker, linestyle="-", color=color, label=label)

    obs.plot_photoz_constraints(redshift=10, ax=ax, marker="o", ms=5, linestyle="none", color="black", capsize=4)
    obs.plot_specz_constraints(redshift=10, ax=ax, marker="s", ms=5, linestyle="none", color="red", capsize=4)

    ax.set_xlabel(r"$M_{\rm UV}$")
    ax.set_ylabel(r"$\log_{10}(\Phi/{\rm Mpc}^{-3}\,{\rm mag}^{-1})$")
    ax.text(0.9, 0.9, r"$z\simeq10$", transform=ax.transAxes, ha="right", va="top", fontsize=20)
    ax.set_xlim(-24, -16)
    ax.set_ylim(-7.5, 0.5)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=12, markerscale=0.6, handlelength=1.5,
              labelspacing=0.4, borderpad=0.4, frameon=True)

    fig.savefig(output_dir / "LCDM_EDE_UVLF_vs_obs_z=10.png", dpi=200, bbox_inches="tight")
    plt.show()

def plot_cv_vs_MUV_limit(lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results,
                         lcdm_C2_results, ede_results, best_b2_sfe_peak):
    fig, ax = plt.subplots(figsize=(8, 6))

    for results, label, marker, color in _series(lcdm_results, lcdm_A1_results, lcdm_C1_results,
                                                 lcdm_B2_results, lcdm_C2_results, ede_results,
                                                 best_b2_sfe_peak):
        ordered = _ordered(results, MUV_limits)
        sigma_cv = np.array([r["sigma_cv"] for r in ordered])
        error_low = np.array([r["sigma_cv_err_low"] for r in ordered])
        error_high = np.array([r["sigma_cv_err_high"] for r in ordered])
        ax.errorbar(MUV_limits, sigma_cv, yerr=[error_low, error_high], marker=marker,
                    linestyle="-", color=color, capsize=4, label=label)

    ax.scatter(MUV_limits, obs_sigma_cv, marker="*", s=220, color="red", label="PANORAMIC obs", zorder=10)
    ax.errorbar(MUV_limits, obs_sigma_cv, yerr=[obs_lower_error, obs_upper_error],
                ecolor="#fc2b26", fmt="none", capsize=6, elinewidth=0.8)

    ax.set_xlabel(r"$M_{\rm UV}$ limit")
    ax.set_ylabel(r"$\sigma_{\rm CV}$")
    ax.text(0.9, 0.9, r"$z\simeq10$", transform=ax.transAxes, ha="right", va="top", fontsize=20)
    ax.set_title(rf"{INSTRUMENT} pencil beams ($M={N_BEAMS}$, $R={N_TRIALS}$)", fontsize=15)
    ax.set_xlim(-19.3, -20.7)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=11, markerscale=0.7, handlelength=1.5, labelspacing=0.4,
              borderpad=0.4, frameon=True)

    fig.savefig(output_dir / "LCDM_EDE_sigma_cv_vs_MUV_limit_z=10.png", dpi=200, bbox_inches="tight")
    plt.show()

def plot_cv_ratio(lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results,
                  lcdm_C2_results, ede_results, best_b2_sfe_peak):
    """Measured sigma_CV over b_eff sigma_m, both box-truncated -- apples to apples."""
    fig, ax = plt.subplots(figsize=(8, 6))

    for results, label, marker, color in _series(lcdm_results, lcdm_A1_results, lcdm_C1_results,
                                                 lcdm_B2_results, lcdm_C2_results, ede_results,
                                                 best_b2_sfe_peak, c2_color="tab:red"):
        ordered = _ordered(results, MUV_limits)
        ratios = np.array([r["cv_ratio"] for r in ordered])
        predicted = np.array([r["sigma_cv_predicted"] for r in ordered])
        error_low = np.array([r["sigma_cv_err_low"] for r in ordered]) / predicted
        error_high = np.array([r["sigma_cv_err_high"] for r in ordered]) / predicted
        ax.errorbar(MUV_limits, ratios, yerr=[error_low, error_high], marker=marker,
                    linestyle="-", color=color, capsize=4, label=label)

    ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5)
    ax.set_xlabel(r"$M_{\rm UV}$ limit")
    ax.set_ylabel(r"$\sigma_{\rm CV}^{\rm measured}/(b_{\rm eff}\sigma_{\rm m,box})$")
    ax.text(0.9, 0.9, r"$z \simeq 10$", transform=ax.transAxes, ha="right", va="top", fontsize=20)
    ax.set_title(rf"{INSTRUMENT} pencil beams ($M={N_BEAMS}$, $R={N_TRIALS}$)", fontsize=15)
    ax.set_xlim(-19.3, -20.7)
    ax.set_ylim(0.75, 3)
    ax.grid(alpha=0.3)
    ax.legend()

    fig.savefig(output_dir / "LCDM_EDE_cv_ratio_z=10.png", dpi=200, bbox_inches="tight")
    plt.show()

def plot_mu_sigma_cv(lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results,
                     lcdm_C2_results, ede_results, best_b2_sfe_peak):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
    series = _series(lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results,
                     lcdm_C2_results, ede_results, best_b2_sfe_peak)

    for plot_id, (ax, MUV_limit) in enumerate(zip(axes, MUV_limits)):
        for results, label, marker, color in series:
            result = next(r for r in results if r["MUV_limit"] == MUV_limit)
            ax.errorbar(result["mu"], result["sigma_cv"],
                        yerr=[[result["sigma_cv_err_low"]], [result["sigma_cv_err_high"]]],
                        xerr=[[result["mu_err_low"]], [result["mu_err_high"]]],
                        marker=marker, color=color, linestyle="none", capsize=3, label=label)

        ax.scatter(obs_mu[plot_id], obs_sigma_cv[plot_id], marker="*", s=220, color="red",
                   label="PANORAMIC obs", zorder=10)
        ax.set_title(rf"$M_{{UV}}<{MUV_limit}$", fontsize=16)
        ax.set_xlabel(r"mean galaxies per field $\mu$")
        ax.grid(alpha=0.3)
        ax.set_xlim(left=0)

    axes[0].set_ylabel(r"$\sigma_{\rm CV}$")
    axes[0].legend(fontsize=9, markerscale=0.8, handlelength=1.2, labelspacing=0.35,
                   borderpad=0.35, frameon=True)
    fig.suptitle(rf"Cosmic variance vs mean counts, {INSTRUMENT} pencil beams at $z\simeq10$", fontsize=20)
    plt.tight_layout()

    fig.savefig(output_dir / "LCDM_EDE_mu_sigma_cv_z=10.png", dpi=200, bbox_inches="tight")
    plt.show()

def plot_pair_covariance(pair_results, model_name="LCDM"):
    """SC96's two branches, measured: Cov(Z_i,Z_j) vs shared volume and vs separation."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    f_centers = 0.5 * (pair_results["f_edges"][:-1] + pair_results["f_edges"][1:])
    r_centers = 0.5 * (pair_results["r_edges"][:-1] + pair_results["r_edges"][1:])

    for limit_id, MUV_limit in enumerate(MUV_limits):
        normalisation = pair_results["diagonal"][limit_id] / pair_results["n_beams"]
        axes[0].plot(f_centers, pair_results["f_profile"][:, limit_id] / normalisation,
                     marker="o", label=rf"$M_{{UV}}<{MUV_limit}$")
        axes[1].plot(r_centers, pair_results["r_profile"][:, limit_id] / normalisation,
                     marker="s", label=rf"$M_{{UV}}<{MUV_limit}$")

    axes[0].set_xlabel("shared volume fraction $f$")
    axes[0].set_ylabel(r"$\mathrm{Cov}(Z_i,Z_j)/\mathrm{Var}(Z)$")
    axes[0].set_title("Overlapping branch", fontsize=15)

    axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1.2)
    axes[1].set_xlabel(r"centre separation $r$ [cMpc]")
    axes[1].set_title("Disjoint branch", fontsize=15)

    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=10)

    fig.suptitle(rf"{model_name}: pair covariance of $Z=(N-\mu)^2$, {INSTRUMENT} beams", fontsize=16)
    plt.tight_layout()

    fig.savefig(output_dir / f"pair_covariance_{model_name.replace(' ', '_')}_z=10.png",
                dpi=200, bbox_inches="tight")
    plt.show()

def plot_saturation(beam_counts, errors, model_name="LCDM"):
    """Where oversampling stops helping: measurement error vs number of beams."""
    fig, ax = plt.subplots(figsize=(7, 5))

    for limit_id, MUV_limit in enumerate(MUV_limits):
        ax.plot(beam_counts, errors[:, limit_id], marker="o", label=rf"$M_{{UV}}<{MUV_limit}$")

    reference = errors[0, 0] * np.sqrt(beam_counts[0] / np.asarray(beam_counts, dtype=float))
    ax.plot(beam_counts, reference, color="black", linestyle="--", linewidth=1.2,
            label=r"$\propto M^{-1/2}$ (independent beams)")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"beams per trial $M$")
    ax.set_ylabel(r"error on $\sigma_{\rm CV}^2$")
    ax.set_title(rf"{model_name}: measurement error saturation, {INSTRUMENT}", fontsize=15)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=10)

    fig.savefig(output_dir / f"saturation_{model_name.replace(' ', '_')}_z=10.png",
                dpi=200, bbox_inches="tight")
    plt.show()

def plot_geometry_comparison(tiling_results, beam_results):
    """Old non-overlapping tiling vs random pencil beams, LCDM baseline only."""
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(MUV_limits, np.array([r["sigma_cv"] for r in _ordered(tiling_results, MUV_limits)]),
            marker="s", color="gray", label="Non-overlapping tiling")

    ordered = _ordered(beam_results, MUV_limits)
    ax.errorbar(MUV_limits, np.array([r["sigma_cv"] for r in ordered]),
                yerr=[np.array([r["sigma_cv_err_low"] for r in ordered]),
                      np.array([r["sigma_cv_err_high"] for r in ordered])],
                marker="o", color="tab:blue", capsize=4, label="Random pencil beams")

    ax.set_xlabel(r"$M_{\rm UV}$ limit")
    ax.set_ylabel(r"$\sigma_{\rm CV}$")
    ax.set_title(rf"{INSTRUMENT} fields at $z\simeq10$", fontsize=15)
    ax.set_xlim(-19.3, -20.7)
    ax.grid(alpha=0.3)
    ax.legend()

    fig.savefig(output_dir / "geometry_comparison_z=10.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# ------------------------------------------------------------------ validation

def tiling_baseline(positions_cmpc, MUV, subhalo_mass, redshift, dims, box_size):
    """The old geometry, run through the old code, in the new result schema."""
    field_ids, inside, _, n_fields = cv.assign_subhalos_to_fields(
        positions_cmpc=positions_cmpc,
        box_size_cmpc=box_size,
        field_volume=np.prod(dims) * u.Mpc ** 3,
        geometry="rectangle",
        field_lengths_cmpc=dims,
    )

    return cv.og_UV_model(
        MUV_inside=MUV[inside],
        subhalo_mass_inside=subhalo_mass[inside],
        redshift=redshift,
        field_ids=field_ids,
        n_fields=n_fields,
        MUV_limits=MUV_limits,
        model_name="LCDM tiling",
    )

def run_validation(positions_cmpc, MUV, dims, box_size, n_beams=200, seed=137):
    """Orientation, containment and mean-count checks on a small set of beams."""
    print("\n=== validation ===")
    check_beam_fits(dims, box_size)

    centers, frames = place_beams(n_beams=n_beams, box_size=box_size, seed=seed)

    for frame in frames:
        assert np.allclose(frame.T @ frame, np.eye(3), atol=1e-10)
        assert np.isclose(np.linalg.det(frame), 1.0, atol=1e-10)
    print("all orientation matrices are proper rotations")

    first_beam_ids = galaxies_in_beam(
        positions=positions_cmpc, center=centers[0], frame=frames[0],
        dims=dims, box_size=box_size,
    )
    assert len(first_beam_ids) == len(np.unique(first_beam_ids))
    print("galaxies inside first beam:", len(first_beam_ids))

    count_matrix = beam_count_matrix(
        positions=positions_cmpc, MUV=MUV, MUV_limits=MUV_limits,
        centers=centers, frames=frames, dims=dims, box_size=box_size, verbose=True,
    )

    assert count_matrix.shape == (n_beams, len(MUV_limits))
    assert np.all(count_matrix >= 0)
    if np.all(np.diff(MUV_limits) < 0):
        assert np.all(np.diff(count_matrix, axis=1) <= 0)

    expected_means, _, _ = validate_mean_counts(
        count_matrix=count_matrix, MUV=MUV, MUV_limits=MUV_limits,
        dims=dims, box_size=box_size,
    )

    # the two estimator forms agree in expectation, and their finite-sample
    # difference is exactly (2 mu - 1)(Nbar - mu) / mu^2 -- check that identity
    central = sigma_cv_squared(count_matrix, expected_means)
    factorial = sigma_cv_squared_factorial(count_matrix, expected_means)
    predicted_difference = (
        (2 * expected_means - 1) * (np.mean(count_matrix, axis=0) - expected_means)
        / expected_means ** 2
    )
    assert np.allclose(factorial - central, predicted_difference)
    print("central form (primary):", central)
    print("factorial form:", factorial)
    print("difference matches (2mu-1)(Nbar-mu)/mu^2:", predicted_difference)

    return count_matrix

In [ ]:
# ------------------------------------------------------------------ main

def main():
    best_b2_sfe_peak = 0.15

    # ---------------------------------------------------------------- LCDM
    lcdm_data = cv.load_data(lcdm_base_path, snap_num)
    lcdm_positions_cmpc, lcdm_MUV, lcdm_subhalo_mass, lcdm_valid = cv.select_valid_uv_galaxies(
        lcdm_data, run_type="LCDM"
    )
    lcdm_box_size = float(lcdm_data["box_size_cmpc"])
    # beams wrap periodically, so every coordinate has to sit in the canonical box
    lcdm_positions_cmpc = np.ascontiguousarray(lcdm_positions_cmpc % lcdm_box_size)
    lcdm_stellar_mass = lcdm_data["stellar_mass_raw"][lcdm_valid] * 1e10 / lcdm_data["h"]

    print("\nLCDM beam geometry")
    lcdm_dims = beam_dimensions(cv.lumina_cosmology, z_center, z_min, z_max, beam_width, beam_height)
    check_beam_fits(lcdm_dims, lcdm_box_size)

    if RUN_VALIDATION:
        run_validation(lcdm_positions_cmpc, lcdm_MUV, lcdm_dims, lcdm_box_size)

    # UV model variants: identical physics to compute_UVLF_cv, applied to the whole box
    lcdm_MUV_A1 = lcdm_MUV - best_delta_MUV

    rng = np.random.default_rng(random_seed)
    lcdm_MUV_C1 = lcdm_MUV + rng.normal(0.0, best_sigma_UV, size=len(lcdm_MUV))

    b2_valid = (
        np.isfinite(lcdm_subhalo_mass) & (lcdm_subhalo_mass > 0)
        & np.isfinite(lcdm_stellar_mass) & (lcdm_stellar_mass > 0)
    )
    b2_max_msun = np.max(lcdm_subhalo_mass[b2_valid]) / lcdm_data["h"]
    print("Lumina B2 pivot mass:", b2_max_msun)

    lcdm_MUV_B2 = cv.B2_DMSFE_MUV(
        MUV=lcdm_MUV, subhalo_mass=lcdm_subhalo_mass, stellar_mass=lcdm_stellar_mass,
        h=lcdm_data["h"], max_halo_mass=b2_max_msun,
        sfe_peak=best_b2_sfe_peak, sfe_slope=best_sfe_slope,
    )
    lcdm_MUV_C2 = cv.C2_mass_dependent_sigma_UV(
        MUV=lcdm_MUV, subhalo_mass=lcdm_subhalo_mass, h=lcdm_data["h"],
        sigma_UV_norm=best_sigma_UV_norm, seed=random_seed,
    )

    # colossus keeps one global cosmology, so it has to be set before any bias call
    cv.which_cosmology("LCDM")

    lcdm_model_MUV = [
        ("LCDM", lcdm_MUV, 0.0),
        ("LCDM A1", lcdm_MUV_A1, best_delta_MUV),
        ("LCDM C1", lcdm_MUV_C1, best_sigma_UV),
        ("LCDM B2", lcdm_MUV_B2, best_b2_sfe_peak),
        ("LCDM C2", lcdm_MUV_C2, best_sigma_UV_norm),
    ]

    lcdm_all_results = []
    lcdm_baseline_trials = None
    for model_name, model_MUV, parameter in lcdm_model_MUV:
        results, trials = beam_UV_model(
            positions=lcdm_positions_cmpc, MUV=model_MUV, subhalo_mass=lcdm_subhalo_mass,
            redshift=lcdm_data["redshift"], MUV_limits=MUV_limits, dims=lcdm_dims,
            box_size=lcdm_box_size, model_name=model_name, parameter=parameter,
        )
        lcdm_all_results.append(results)
        if model_name == "LCDM":
            lcdm_baseline_trials = trials

    lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results, lcdm_C2_results = lcdm_all_results

    print("\nLCDM finite-volume term (quoted separately, never in the error bar)")
    lcdm_cosmology = cv.which_cosmology("LCDM")
    lcdm_sigma_m = cv.compute_sigma_m(
        cosmology=lcdm_cosmology, field_lengths_cmpc=lcdm_dims,
        box_size_cmpc=lcdm_box_size, h=lcdm_data["h"], redshift=lcdm_data["redshift"],
    )
    lcdm_missing = sigma_m_missing_modes(
        cosmology=lcdm_cosmology, dims=lcdm_dims, box_size=lcdm_box_size,
        h=lcdm_data["h"], redshift=lcdm_data["redshift"],
    )
    print("LCDM box-mode sigma_m:", lcdm_sigma_m)
    # independent reimplementation of the same lattice sum, as a cross-check
    print("LCDM box-mode sigma_m (independent sum):", np.sqrt(sigma_m_modes(
        cosmology=lcdm_cosmology, dims=lcdm_dims, box_size=lcdm_box_size,
        h=lcdm_data["h"], redshift=lcdm_data["redshift"],
    )))

    # ---------------------------------------------------------------- EDE
    ede_data = cv.load_EDE_data(
        hdf5_path=ede_hdf5_path, redshift=redshift_EDE, box_size_cmpc=box_size_cmpc_EDE,
    )
    ede_positions_cmpc, ede_MUV, ede_subhalo_mass, ede_valid = cv.select_valid_uv_galaxies(
        ede_data, run_type="EDE"
    )
    ede_box_size = float(ede_data["box_size_cmpc"])
    ede_positions_cmpc = np.ascontiguousarray(ede_positions_cmpc % ede_box_size)

    print("\nEDE beam geometry")
    ede_dims = beam_dimensions(cv.ede_cosmology, z_center, z_min, z_max, beam_width, beam_height)
    check_beam_fits(ede_dims, ede_box_size)

    cv.which_cosmology("EDE")
    ede_results, ede_trials = beam_UV_model(
        positions=ede_positions_cmpc, MUV=ede_MUV, subhalo_mass=ede_subhalo_mass,
        redshift=ede_data["redshift"], MUV_limits=MUV_limits, dims=ede_dims,
        box_size=ede_box_size, model_name="EDE",
    )

    print("\nEDE finite-volume term")
    ede_cosmology = cv.which_cosmology("EDE")
    ede_sigma_m = cv.compute_sigma_m(
        cosmology=ede_cosmology, field_lengths_cmpc=ede_dims,
        box_size_cmpc=ede_box_size, h=ede_data["h"], redshift=ede_data["redshift"],
    )
    ede_missing = sigma_m_missing_modes(
        cosmology=ede_cosmology, dims=ede_dims, box_size=ede_box_size,
        h=ede_data["h"], redshift=ede_data["redshift"],
    )
    print("EDE box-mode sigma_m:", ede_sigma_m)

    # ------------------------------------------------- measured vs predicted CV
    for model_results, sigma_m, missing in [
        (lcdm_results + lcdm_A1_results + lcdm_C1_results + lcdm_B2_results + lcdm_C2_results,
         lcdm_sigma_m, lcdm_missing),
        (ede_results, ede_sigma_m, ede_missing),
    ]:
        for result in model_results:
            result["sigma_m"] = sigma_m
            result["sigma_m_missing_squared"] = missing
            # ratio uses the box-truncated sigma_m: measurement and prediction
            # are then both missing the same modes, which is the fair comparison
            result["sigma_cv_predicted"] = result["b_eff"] * sigma_m
            prediction = result["sigma_cv_predicted"]
            result["cv_ratio"] = (
                result["sigma_cv"] / prediction
                if np.isfinite(prediction) and prediction > 0 else np.nan
            )
            # universe-level statement, linear response, quoted separately
            result["sigma_cv_with_missing_modes"] = np.sqrt(
                max(result["sigma_cv"] ** 2 + result["b_eff"] ** 2 * missing, 0.0)
            )

    print("\nfinite-volume correction (linear response, assumption 8)")
    for result in _ordered(lcdm_results, MUV_limits):
        print(f"  LCDM M_UV < {result['MUV_limit']}: "
              f"sigma_cv(box)={result['sigma_cv']:.4g} -> "
              f"sigma_cv(+missing modes)={result['sigma_cv_with_missing_modes']:.4g}")

    # ---------------------------------------------------------------- plots
    plot_uvlf_overlay(
        lcdm_MUV=lcdm_MUV, lcdm_MUV_A1=lcdm_MUV_A1, lcdm_MUV_C1=lcdm_MUV_C1,
        lcdm_MUV_B2=lcdm_MUV_B2, lcdm_MUV_C2=lcdm_MUV_C2, ede_MUV=ede_MUV,
        lcdm_box_size=lcdm_box_size, ede_box_size=ede_box_size,
        best_b2_sfe_peak=best_b2_sfe_peak,
    )
    plot_cv_vs_MUV_limit(lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results,
                         lcdm_C2_results, ede_results, best_b2_sfe_peak)
    plot_mu_sigma_cv(lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results,
                     lcdm_C2_results, ede_results, best_b2_sfe_peak)
    plot_cv_ratio(lcdm_results, lcdm_A1_results, lcdm_C1_results, lcdm_B2_results,
                  lcdm_C2_results, ede_results, best_b2_sfe_peak)

    # ------------------------------------------------- pair covariance and saturation
    lcdm_pairs = ede_pairs = None
    if RUN_PAIR_ANALYSIS:
        lcdm_pairs = pair_covariance_analysis(
            positions=lcdm_positions_cmpc, MUV=lcdm_MUV, MUV_limits=MUV_limits,
            dims=lcdm_dims, box_size=lcdm_box_size, model_name="LCDM",
        )
        plot_pair_covariance(lcdm_pairs, model_name="LCDM")

        ede_pairs = pair_covariance_analysis(
            positions=ede_positions_cmpc, MUV=ede_MUV, MUV_limits=MUV_limits,
            dims=ede_dims, box_size=ede_box_size, model_name="EDE",
        )
        plot_pair_covariance(ede_pairs, model_name="EDE")

        # cross-check: trial scatter vs the pair-sum reconstruction of the same quantity
        print("\ncross-check, Var(sigma_CV^2) from trials vs from the pair sum (LCDM):")
        trial_variance = lcdm_baseline_trials["sigma_cv_squared_error"] ** 2
        pair_variance = lcdm_pairs["sigma_cv_squared_variance"]
        for limit_id, MUV_limit in enumerate(MUV_limits):
            print(f"  M_UV < {MUV_limit}: trials={trial_variance[limit_id]:.5g}, "
                  f"pairs={pair_variance[limit_id]:.5g}, "
                  f"ratio={pair_variance[limit_id] / trial_variance[limit_id]:.3f}")
        print("  (the two use different M, so compare M_eff and the branch split, "
              "not the raw variances)")

    if RUN_SATURATION:
        print("\nLCDM measurement-error saturation")
        beam_counts, errors, effective = saturation_curve(
            positions=lcdm_positions_cmpc, MUV=lcdm_MUV, MUV_limits=MUV_limits,
            dims=lcdm_dims, box_size=lcdm_box_size, model_name="LCDM",
        )
        plot_saturation(beam_counts, errors, model_name="LCDM")

    if RUN_VALIDATION:
        cv.which_cosmology("LCDM")
        tiling_results = tiling_baseline(
            positions_cmpc=lcdm_positions_cmpc, MUV=lcdm_MUV,
            subhalo_mass=lcdm_subhalo_mass, redshift=lcdm_data["redshift"],
            dims=lcdm_dims, box_size=lcdm_box_size,
        )
        plot_geometry_comparison(tiling_results, lcdm_results)

    return {
        "LCDM": lcdm_results,
        "LCDM A1": lcdm_A1_results,
        "LCDM C1": lcdm_C1_results,
        "LCDM B2": lcdm_B2_results,
        "LCDM C2": lcdm_C2_results,
        "EDE": ede_results,
        "LCDM trials": lcdm_baseline_trials,
        "EDE trials": ede_trials,
        "LCDM pairs": lcdm_pairs,
        "EDE pairs": ede_pairs,
    }

In [ ]:
if __name__ == "__main__":
    all_results = main()